# Diabetes Health Indicators — Tabular ResNet18 Quantum-Gated Model

This notebook is designed for the Kaggle **Diabetes Health Indicators Dataset**.

**Default file:** `diabetes_binary_5050split_health_indicators_BRFSS2015.csv`

Why a tabular ResNet instead of torchvision ResNet18?
- torchvision ResNet18 is a 2D image CNN and expects image tensors such as `(B, 3, 224, 224)`.
- This dataset has **21 structured health-indicator features**, so converting rows into fake images would not be scientifically meaningful.
- The model below preserves the **ResNet18 residual-block pattern (2,2,2,2)** in a tabular network.
- This gives us a fair classical baseline that can be matched exactly by the quantum-gated version.

The train/validation/test protocol is fixed at **70/15/15**, stratified by target, with `StandardScaler` fitted on the training split only.


## Quantum adaptation

The uploaded image notebook uses:

`ResNet layer3 features -> squeeze -> 2 qubits -> variational circuit -> sigmoid gate -> multiply layer3 channels -> layer4 -> classifier`

This notebook preserves that same logic for tabular residual features:

`Tabular ResNet layer3 (256 features) -> squeeze -> 2 qubits -> quantum circuit -> 256-feature sigmoid gate -> layer4 -> classifier`

The classifier still receives the full 512-dimensional classical representation. The two quantum values control a gate; they are **not** the final classifier bottleneck.

In [ ]:
%pip install -q pennylane

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

import pennylane as qml

from pathlib import Path
from datetime import datetime, timezone

SEED = 42
BATCH_SIZE = 256
EPOCHS = 30
LR = 1e-3
PATIENCE = 5
NUM_WORKERS = 2

N_QUBITS = 2
N_QLAYERS = 2

CSV_NAME = "diabetes_binary_5050split_health_indicators_BRFSS2015.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Classical device:", DEVICE)
print("PennyLane:", qml.__version__)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

RUN_DIR = Path("/kaggle/working") / (
    "diabetes_quantum_gated_tabular_resnet18_" +
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", RUN_DIR)

## 1. Load the selected CSV

In [ ]:
matches = list(Path("/kaggle/input").rglob(CSV_NAME))
if not matches:
    raise FileNotFoundError(
        f"Could not find {CSV_NAME}. Add the Diabetes Health Indicators dataset to this Kaggle notebook."
    )

CSV_PATH = matches[0]
df = pd.read_csv(CSV_PATH)

print("CSV:", CSV_PATH)
print("Shape:", df.shape)
display(df.head())
print("Missing values:", int(df.isna().sum().sum()))

## 2. Detect target and prepare features

In [ ]:
if "Diabetes_binary" in df.columns:
    TARGET = "Diabetes_binary"
elif "Diabetes_012" in df.columns:
    TARGET = "Diabetes_012"
else:
    raise ValueError("Expected target column Diabetes_binary or Diabetes_012.")

X_df = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int).to_numpy()

if not all(np.issubdtype(dtype, np.number) for dtype in X_df.dtypes):
    bad = X_df.select_dtypes(exclude=[np.number]).columns.tolist()
    raise ValueError(f"Non-numeric columns found: {bad}")

feature_names = X_df.columns.tolist()
X = X_df.to_numpy(dtype=np.float32)

classes = np.unique(y)
NUM_CLASSES = len(classes)
if not np.array_equal(classes, np.arange(NUM_CLASSES)):
    raise ValueError(f"Expected class labels 0..K-1, got {classes}")

print("Target:", TARGET)
print("Number of features:", X.shape[1])
print("Classes:", classes)
print(pd.Series(y).value_counts().sort_index())

## 3. Same stratified 70/15/15 split and train-only scaling as baseline

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

In [ ]:
def make_loader(X_arr, y_arr, shuffle):
    ds = TensorDataset(
        torch.from_numpy(X_arr),
        torch.from_numpy(y_arr).long()
    )
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

train_loader = make_loader(X_train, y_train, True)
val_loader = make_loader(X_val, y_val, False)
test_loader = make_loader(X_test, y_test, False)
print("Batches:", len(train_loader), len(val_loader), len(test_loader))

## 4. Two-qubit variational quantum circuit — same idea as uploaded model

In [ ]:
q_device = qml.device("default.qubit", wires=N_QUBITS)

def _circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

qnode = qml.QNode(
    _circuit,
    q_device,
    interface="torch",
    diff_method="backprop"
)

weight_shapes = {"weights": (N_QLAYERS, N_QUBITS)}

def make_quantum_layer():
    return qml.qnn.TorchLayer(qnode, weight_shapes)

print("Quantum circuit ready:", N_QUBITS, "qubits,", N_QLAYERS, "layers")

## 5. Quantum-gated Tabular ResNet18

In [ ]:
class ResidualMLPBlock(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim)
        self.bn1 = nn.BatchNorm1d(out_dim)
        self.fc2 = nn.Linear(out_dim, out_dim)
        self.bn2 = nn.BatchNorm1d(out_dim)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Identity() if in_dim == out_dim else nn.Linear(in_dim, out_dim)

    def forward(self, x):
        identity = self.skip(x)
        out = self.fc1(x)
        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.bn2(out)
        return F.relu(out + identity)


class QuantumGatedTabularResNet18(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.layer1 = nn.Sequential(
            ResidualMLPBlock(64, 64),
            ResidualMLPBlock(64, 64)
        )
        self.layer2 = nn.Sequential(
            ResidualMLPBlock(64, 128),
            ResidualMLPBlock(128, 128)
        )
        self.layer3 = nn.Sequential(
            ResidualMLPBlock(128, 256),
            ResidualMLPBlock(256, 256)
        )

        # Mirrors uploaded image notebook:
        # 256 intermediate channels -> squeeze to 2 -> quantum -> excite back to 256
        self.squeeze = nn.Sequential(
            nn.Linear(256, 32),
            nn.ReLU(),
            nn.Linear(32, N_QUBITS),
            nn.Tanh()
        )
        self.quantum = make_quantum_layer()
        self.excite = nn.Sequential(
            nn.Linear(N_QUBITS, 256),
            nn.Sigmoid()
        )

        self.layer4 = nn.Sequential(
            ResidualMLPBlock(256, 512),
            ResidualMLPBlock(512, 512)
        )
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)             # (B, 256)

        q_in = self.squeeze(x) * np.pi

        # PennyLane default.qubit is run on CPU for broad Kaggle compatibility.
        q_out = self.quantum(
            q_in.to(device="cpu", dtype=torch.float32)
        )
        q_out = q_out.to(device=x.device, dtype=x.dtype)

        gate = self.excite(q_out)       # (B, 256), values in [0, 1]
        x = x * gate

        x = self.layer4(x)
        return self.head(x)


model = QuantumGatedTabularResNet18(
    input_dim=X_train.shape[1],
    num_classes=NUM_CLASSES
).to(DEVICE)

print(model)

## 6. Forward/backward smoke test, including quantum gradients

In [ ]:
model.train()
smoke_x = torch.randn(8, X_train.shape[1], device=DEVICE)
smoke_y = torch.randint(0, NUM_CLASSES, (8,), device=DEVICE)

smoke_logits = model(smoke_x)
assert smoke_logits.shape == (8, NUM_CLASSES)

smoke_loss = nn.CrossEntropyLoss()(smoke_logits, smoke_y)
smoke_loss.backward()

for name, module in [
    ("layer3", model.layer3),
    ("squeeze", model.squeeze),
    ("quantum", model.quantum),
    ("excite", model.excite),
    ("layer4", model.layer4),
    ("head", model.head),
]:
    grads = [p.grad for p in module.parameters() if p.requires_grad]
    assert grads, f"No trainable parameters in {name}"
    assert all(g is not None for g in grads), f"Missing gradient in {name}"
    assert all(torch.isfinite(g).all() for g in grads), f"Non-finite gradient in {name}"

model.zero_grad(set_to_none=True)
print("PASS: classical and quantum-gating branches receive gradients.")

## 7. Metrics

In [ ]:
def compute_metrics(labels, probs):
    preds = probs.argmax(axis=1)

    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

    if NUM_CLASSES == 2:
        metrics["roc_auc"] = roc_auc_score(labels, probs[:, 1])

        cm = confusion_matrix(labels, preds, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        metrics["sensitivity"] = tp / (tp + fn) if (tp + fn) else float("nan")
        metrics["specificity"] = tn / (tn + fp) if (tn + fp) else float("nan")
    else:
        try:
            metrics["roc_auc_macro_ovr"] = roc_auc_score(
                labels, probs, multi_class="ovr", average="macro"
            )
        except ValueError:
            metrics["roc_auc_macro_ovr"] = float("nan")

    return metrics

## 8. Training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    probs_list, labels_list = [], []

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        probs_list.append(F.softmax(logits, dim=1).cpu().numpy())
        labels_list.append(yb.cpu().numpy())

    probs = np.concatenate(probs_list)
    labels = np.concatenate(labels_list)

    metrics = {"loss": total_loss / len(loader.dataset)}
    metrics.update(compute_metrics(labels, probs))
    return metrics, probs, labels


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_f1 = -1.0
best_state = None
best_epoch = 0
stale = 0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_metrics, _, _ = evaluate(model, val_loader, criterion)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()}
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_f1={val_metrics['f1_macro']:.4f} | "
        f"{time.time()-t0:.1f}s"
    )

    if val_metrics["f1_macro"] > best_f1 + 1e-4:
        best_f1 = val_metrics["f1_macro"]
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(best_state)
torch.save(best_state, RUN_DIR / "quantum_gated_tabular_resnet18.pt")
pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)

print("Best epoch:", best_epoch)
print("Best validation macro-F1:", best_f1)

## 9. Final validation and held-out test results

In [ ]:
results = []

for split_name, loader in [("validation", val_loader), ("test", test_loader)]:
    metrics, probs, labels = evaluate(model, loader, criterion)
    preds = probs.argmax(axis=1)

    print(f"\n=== {split_name.upper()} ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    print("\nClassification report:")
    print(classification_report(labels, preds, digits=4, zero_division=0))

    results.append({
        "model": "Quantum-gated Tabular ResNet18",
        "split": split_name,
        **metrics
    })

    pd.DataFrame({
        "true_label": labels,
        "predicted_label": preds,
        **{f"prob_class_{i}": probs[:, i] for i in range(NUM_CLASSES)}
    }).to_csv(RUN_DIR / f"{split_name}_predictions.csv", index=False)

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(RUN_DIR / "metrics.csv", index=False)

## 10. Inspect learned quantum-controlled gates

In [ ]:
gate_capture = {}

def gate_hook(module, inputs, output):
    gate_capture["gate"] = output.detach().cpu()

handle = model.excite.register_forward_hook(gate_hook)

model.eval()
xb, yb = next(iter(val_loader))
with torch.no_grad():
    _ = model(xb.to(DEVICE))

handle.remove()

gates = gate_capture["gate"].numpy()
print("Gate tensor shape:", gates.shape)
print("Mean gate value:", gates.mean())
print("Std gate value:", gates.std())
print("Min / Max:", gates.min(), gates.max())

plt.figure(figsize=(7, 4))
plt.hist(gates.ravel(), bins=40)
plt.xlabel("Sigmoid gate value")
plt.ylabel("Count")
plt.title("Quantum-controlled feature-gate distribution")
plt.tight_layout()
plt.show()

## 11. Training curves

In [ ]:
hist = pd.DataFrame(history)

plt.figure(figsize=(7, 4))
plt.plot(hist["epoch"], hist["train_loss"], label="train loss")
plt.plot(hist["epoch"], hist["val_loss"], label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(hist["epoch"], hist["val_f1_macro"])
plt.xlabel("Epoch")
plt.ylabel("Validation macro-F1")
plt.tight_layout()
plt.show()

## 12. Export deployment artifacts\n\nExports the restored best-validation checkpoint, exact train-fitted scaler, and backend reconstruction metadata.\n

In [ ]:
# Export the restored best-validation model and the exact train-fitted scaler.
import joblib
import shutil
from IPython.display import FileLink, display

DEPLOY_DIR = Path("/kaggle/working/diabetes_backend_model")
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = DEPLOY_DIR / "diabetes_model.pth"
SCALER_PATH = DEPLOY_DIR / "diabetes_scaler.pkl"
CONFIG_PATH = DEPLOY_DIR / "diabetes_config.json"
ZIP_PATH = Path("/kaggle/working/diabetes_backend_model.zip")

# `model` was restored from best_state selected by validation macro-F1.
torch.save(model.state_dict(), MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)

# Keep exactly the labels detected by the existing notebook; do not remap them.
class_mapping = {str(int(class_id)): int(class_id) for class_id in classes}

config = {
    "model_name": "QuantumGatedTabularResNet18",
    "architecture_name": "Quantum-gated Tabular ResNet18-style residual MLP [2,2,2,2]",
    "feature_names": feature_names,
    "target_column": TARGET,
    "input_dimension": len(feature_names),
    "num_classes": NUM_CLASSES,
    "class_mapping": class_mapping,
    "scaler_type": "StandardScaler",
    "n_qubits": N_QUBITS,
    "n_quantum_layers": N_QLAYERS,
    "dropout": 0.10,
    "seed": SEED,
    "split": "stratified 70/15/15",
    "pennylane_device": "default.qubit",
    "quantum_embedding_type": "qml.AngleEmbedding",
    "quantum_layer_type": "qml.BasicEntanglerLayers",
    "quantum_expectations": "PauliZ expectation values",
    "output_type": "raw logits",
    "inference_probability_operation": "softmax",
    "best_epoch": best_epoch,
    "best_validation_macro_f1": best_f1,
    "pytorch_version": torch.__version__,
    "pennylane_version": qml.__version__,
}
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=DEPLOY_DIR.parent,
    base_dir=DEPLOY_DIR.name,
)

print("Deployment files:")
for path in (MODEL_PATH, SCALER_PATH, CONFIG_PATH, ZIP_PATH):
    print(path.resolve())
print(f"Download this ZIP from Kaggle: {ZIP_PATH.resolve()}")
display(FileLink(ZIP_PATH, result_html_prefix="Kaggle download link: "))


## Deployment verification smoke test\n

In [ ]:
# Deployment smoke test: rebuild the exact quantum model and load CPU artifacts.
reloaded_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
reloaded_scaler = joblib.load(SCALER_PATH)

# Cell 11's _circuit, qnode, and make_quantum_layer, plus cell 13's model
# classes, remain the exact architecture definitions used for training.
reloaded_model = QuantumGatedTabularResNet18(
    input_dim=reloaded_config["input_dimension"],
    num_classes=reloaded_config["num_classes"],
)
reloaded_model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
reloaded_model.eval()

# Start from one raw dataset row and apply the deployment preprocessing exactly.
raw_sample = df.iloc[[0]].copy()
sample_features = raw_sample.drop(columns=[reloaded_config["target_column"]])
sample_features = sample_features.loc[:, reloaded_config["feature_names"]]
scaled_sample = reloaded_scaler.transform(
    sample_features.to_numpy(dtype=np.float32)
).astype(np.float32)
sample_tensor_cpu = torch.from_numpy(scaled_sample).to(dtype=torch.float32)

model.eval()
with torch.no_grad():
    original_logits = model(sample_tensor_cpu.to(DEVICE)).cpu()
    reloaded_logits = reloaded_model(sample_tensor_cpu)
    probabilities = torch.softmax(reloaded_logits, dim=1)

np.testing.assert_allclose(
    original_logits.numpy(), reloaded_logits.numpy(), rtol=1e-5, atol=1e-6
)
predicted_class = int(probabilities.argmax(dim=1).item())
print("Smoke test passed: reloaded CPU model matches the restored best model.")
print("Predicted class:", predicted_class)
for class_index, probability in enumerate(probabilities[0].tolist()):
    print(f"Probability of class {class_index}: {probability}")
